# GSE150062 processing decisions

Executable reconstruction record for task `t_680f05a3`. This notebook binds the source identities, validates the append-only production receipts, and records the strict VAR boundary without inventing Ensembl identifiers for source-native `LH` features.

In [ ]:
import json
from pathlib import Path

root = Path.cwd()
evidence = root / 'artifacts/schema_audit/real_dataset_curation_20260722/geo_GSE150062/t_680f05a3'
source = json.loads((evidence / 'source_manifest.json').read_text())
inspection = json.loads((evidence / 'inspection_report.json').read_text())
plan = json.loads((evidence / 'plan_receipt.json').read_text())
mutation = json.loads((evidence / 'mutation_receipt.json').read_text())
verification = json.loads((evidence / 'verification_receipt.json').read_text())


## Accepted source and axis facts

Table S5 is the exact accepted-cell authority. The GEO expression axis and the immutable author-reference commit are the feature authorities. Cell Ranger's documented underscore-to-hyphen sanitization accounts for 29 feature spelling differences.

In [ ]:
assert source['dataset']['accepted_observations'] == 78_393
assert source['dataset']['accepted_expression_features'] == 60_497
assert source['table_s5']['sha256'] == 'fd307e6aec5a0044e0ec135594ed1d3071d3efb609809faa1f0ef91d111b465c'
assert source['author_code']['commit'] == '4285c7f5e81eaada87cf668b7fa4039f4ff3b1c9'
assert inspection['current']['x_axis']['shape'] == [78_393, 60_497]
assert inspection['current']['x_axis']['obs_names_equal_table']
assert inspection['axis_relations']['original_obs_index']['ordered_equals_table_s5']


## Publication and immutable readback

The mutation created one OBS revision and one VAR revision, changed neither X nor Collections, and deleted nothing. The independent verify-mode replay read the immutable identities back and made zero writes.

In [ ]:
assert mutation['status'] == 'PASS' and mutation['mode'] == 'mutate'
assert mutation['writes']['obs_revisions'] == 1
assert mutation['writes']['var_revisions'] == 1
assert mutation['writes']['x_revisions'] == 0
assert mutation['writes']['collection_writes'] == 0
assert mutation['writes']['deletions'] == 0
assert mutation['writes']['artifacts']['obs'][0]['uid'] == 'CkcQf1IYkOkbxKed0003'
assert mutation['writes']['artifacts']['var'][0]['uid'] == 'rRlvtvSEpbFnek7K0002'
assert verification['status'] == 'PASS' and verification['mode'] == 'verify'
assert verification['replay_noop']
assert verification['registry_counts']['before'] == verification['registry_counts']['after']
assert verification['member_after']['obs_before']['hash'] == '_yy1Dn0rGA84AVtobdWucA'
assert verification['member_after']['var_before']['hash'] == 'FP_t3gG44JtNfOQwoF1bGw'


## Contract verdicts and non-fabrication boundary

`OBS_COMPLETED` is supported by source-exhaustive Table S5 semantics and exact row/order preservation. The strict literal `VAR_ENSEMBL_SPECIES_COMPLETED` criterion remains false: 16,401 biological features use the authors' source-native `LH` identifiers and 71 more are ambiguous or noncanonical. All 60,497 features have exact source feature identity and human organism provenance, but those facts do not license fabricated ENSG assignments. Consequently `DATASET_E2E_V3` is blocked only on the strict VAR criterion.

In [ ]:
e2e = verification['dataset_e2e_v3']
var = verification['member_after']['var_verdict']
assert e2e['obs_completed'] is True
assert e2e['var_ensembl_species_completed'] is False
assert e2e['status'] == 'blocked_var_source_identifier_boundary'
assert var['biological_features_total'] == 60_497
assert var['stable_ensembl_id_features'] == 44_025
assert var['source_native_lh_features'] == 16_401
assert var['other_nonpassing_features'] == 71
assert var['correct_species_features'] == 60_497
print(json.dumps({
    'obs_uid': verification['member_after']['obs_before']['uid'],
    'var_uid': verification['member_after']['var_before']['uid'],
    'obs_completed': e2e['obs_completed'],
    'var_ensembl_species_completed': e2e['var_ensembl_species_completed'],
    'dataset_e2e_status': e2e['status'],
    'replay_noop': verification['replay_noop'],
}, indent=2, sort_keys=True))
